# The Seasons — a life-transitions guide

A demo of building a persona-driven conversational agent on the dialectical framework, themed on change as a cycle of seasons.

This notebook drives the **Advisor** agent — a pure-conversation agent where the
dialectical framework runs silently in the background. You never see framework
terminology; the persona is defined entirely by an **app preamble** (a "persona
skin") written inline below. Same dialectical engine, different voice — see
`src/dialectical_framework/agents/apps.py` for the shipped preambles and the
guide to authoring your own.

## Prerequisites

1. **Start Memgraph** (the Advisor builds a graph behind the scenes):
   run `/df-memgraph start`, or `docker compose -f docker-compose.test.yml up -d`.
2. **Configure the LLM**: copy `.env.example` to `.env` and set
   `DIALEXITY_DEFAULT_MODEL` (e.g. `bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0`)
   plus the matching provider credentials (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, or AWS creds).
3. **Run the cells top to bottom.** Jupyter supports top-level `await`, so the async
   `advisor.chat(...)` calls work directly.

## 1. Bootstrap

`DialecticalReasoning.setup(...)` builds and auto-wires the DI container (this is
the one line the host app normally runs at startup). A committed `Case` owns the
`sid` scope that every graph write is bound to.

In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.agents.advisor.advisor import Advisor

# Build + auto-wire the DI container once (reads .env via Settings.from_env()).
DialecticalReasoning.setup(Settings.from_env())

# A Case owns the sid scope that all graph writes are bound to.
case = Case()
case.commit()
print(f"Case ready — sid={case.sid}")

## 2. The persona: a seasons guide

Change moves in seasons — summer's fullness gives way to autumn's release, winter's
fallow stretch, then spring's renewal. People navigating a transition are usually
stuck between two of these seasons: clinging to a summer that's ending, or dreading
a winter they're already in.

The preamble below turns the Advisor into a guide for life transitions. It shapes
**only** voice and delivery — how endings, fallow periods, and beginnings are named.
The dialectical engine underneath does the structural work of surfacing what the
person can't yet see about the season they're resisting.

In [ ]:
SEASONS_GUIDE_APP = """## Persona

You are a guide for people in transition. You understand change as seasonal: every
ending (autumn) makes room, every fallow stretch (winter) does invisible work, and
every beginning (spring) grows from what was composted before it. You help people
locate which season they are actually in — and which one they are resisting.

When someone is gripping a summer that's ending, you name — kindly but plainly — the
cost of refusing autumn, and the harvest that only release makes possible. When
someone is stuck in a winter and reading it as failure, you reveal the quiet, generative
work that fallow time is doing. You present these as discoveries about the natural
shape of their situation, not as criticism of how they've handled it.

When you offer a way forward, you frame it as the next seasonal move — what to let
fall, what to let rest, what to plant — always as an option with real tradeoffs, never
a prescription. You never rush someone toward spring before their winter is done.

Your tone is warm, grounded, and patient, with a long view of time. You use seasonal
and natural imagery lightly, in service of clarity — never as decoration.
"""

## 3. Chat with the guide

All chat happens inside `with scope(case.sid):` so every graph write lands in this
Case. The `Advisor` is constructed with our inline preamble; `chat()` is async and
returns the assistant's text. History is retained on `advisor.messages`, so the
follow-up turn builds on the first.

In [ ]:
with scope(case.sid):
    advisor = Advisor(app_preamble=SEASONS_GUIDE_APP)
    reply = await advisor.chat(
        "My kids just left for college and the house is silent. Everyone says "
        "'enjoy the freedom,' but I mostly feel useless, like my main job just "
        "ended and nothing has replaced it."
    )
print(reply)

In [ ]:
# Follow-up turn — same scope, same advisor, history preserved.
with scope(case.sid):
    reply = await advisor.chat(
        "I keep trying to fill the time to feel productive again, but it all "
        "feels forced. Part of me suspects I should just let it be empty for a while."
    )
print(reply)

## 4. Try your own

- **Swap the persona.** Feel the same engine in a different voice:
  ```python
  from dialectical_framework.agents.apps import COUNSELOR_APP, STRATEGIC_ADVISOR_APP
  advisor = Advisor(app_preamble=STRATEGIC_ADVISOR_APP)
  ```
- **Inspect the running conversation** with `advisor.messages`.
- **Resume later** by passing saved messages: `Advisor(app_preamble=SEASONS_GUIDE_APP, messages=saved)`.
- **Start from an existing analysis** by precomputing context:
  ```python
  from dialectical_framework.concerns.dialectical_context import DialecticalContext
  with scope(case.sid):
      context = await DialecticalContext().resolve()
      advisor = Advisor(app_preamble=SEASONS_GUIDE_APP, dialectical_context=context)
  ```